# Model Training 

In this phase, we train multiple machine learning models to predict customer churn and identify high-risk customers. The objective is to build a robust predictive system by comparing different algorithms and selecting the best-performing model based on evaluation metrics.

##  Objectives of This Phase

- Train multiple classification models for churn prediction  
- Establish a baseline model for performance comparison  
- Improve prediction accuracy using advanced ensemble models  
- Evaluate models using standard classification metrics  
- Select the best model for production-level deployment  

##  Models Used

To ensure a strong and reliable prediction system, we use the following models:

### 1️⃣ Logistic Regression (Baseline Model)
A simple linear model used as a performance benchmark.

### 2️⃣ XGBoost Classifier
A powerful gradient boosting algorithm known for high accuracy and performance on structured data.

### 3️⃣ LightGBM Classifier
A fast and efficient gradient boosting model optimized for large datasets and faster training.


In [1]:
import sys
import os

PROJECT_ROOT = r"c:\Projects\TEYZIX-CORE-INTERNSHIP\task-3_Telco_Churn_System"
sys.path.append(PROJECT_ROOT)

In [2]:
import pandas as pd
import numpy as  np
data=pd.read_csv(r"../dataset/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [3]:
data.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
data.drop("customerID", axis=1, inplace=True)

In [5]:
data["TotalCharges"] = pd.to_numeric(
    data["TotalCharges"],
    errors="coerce"
)

In [6]:
data["TotalCharges"].fillna(
    data["TotalCharges"].median(),
    inplace=True
)

C:\Users\10\AppData\Local\Temp\ipykernel_17132\2560722785.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data["TotalCharges"].fillna(


In [7]:
data["Churn"] = data["Churn"].map({
    "Yes":1,
    "No":0
})

In [8]:
print(data.dtypes)

gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                 int64
dtype: object


In [9]:
data = pd.get_dummies(data, drop_first=True)

In [10]:
data.shape

(7043, 31)

In [11]:
X = data.drop("Churn", axis=1)
y = data["Churn"]

In [31]:
import joblib
import os

BASE_DIR = os.path.dirname(os.getcwd())

MODELS_DIR = os.path.join(BASE_DIR, "models")

os.makedirs(MODELS_DIR, exist_ok=True)

FEATURE_PATH = os.path.join(
    MODELS_DIR,
    "model_features.pkl"
)

joblib.dump(list(X.columns), FEATURE_PATH)

print("model_features.pkl saved successfully")
print(FEATURE_PATH)

model_features.pkl saved successfully
c:\Projects\TEYZIX-CORE-INTERNSHIP\task-3_Telco_Churn_System\models\model_features.pkl


In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

##  1. Logistic Regression

### How it Works:
Logistic Regression is a **linear classification algorithm** that predicts the probability of a binary outcome (churn or not churn). It uses the sigmoid function to convert linear outputs into probabilities between 0 and 1.

### Importance in This Project:
- Serves as a **baseline model** for comparison  
- Simple, fast, and easy to interpret  
- Helps understand basic relationships between features and churn  
- Provides explainable coefficients for business interpretation  

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Logistic Regression Results:")
print(classification_report(y_test, y_pred_log))
print("AUC:", roc_auc_score(y_test, log_model.predict_proba(X_test)[:,1]))

Logistic Regression Results:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.55      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.74      1409
weighted avg       0.80      0.80      0.80      1409

AUC: 0.8424629931023793


C:\Users\10\AppData\Roaming\Python\Python310\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


##  2. XGBoost (Extreme Gradient Boosting)

###  How it Works:
XGBoost is an **advanced ensemble learning algorithm** that builds multiple decision trees sequentially. Each new tree tries to correct the errors of the previous ones using gradient boosting.

###  Importance in This Project:
- High accuracy on structured/tabular data  
- Handles complex feature interactions automatically  
- Reduces overfitting using regularization techniques  
- Efficient and widely used in real-world ML systems  

### Why We Use It:
XGBoost is often one of the **best-performing algorithms for churn prediction problems**, making it a strong candidate for production use.

In [14]:
from xgboost import XGBClassifier

In [15]:
xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric='logloss'
)

In [16]:
xgb_model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [17]:
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:,1]

In [18]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

print("XGBoost Results:")
print(classification_report(y_test, y_pred_xgb))

print("AUC Score:", roc_auc_score(y_test, y_prob_xgb))

XGBoost Results:
              precision    recall  f1-score   support

           0       0.84      0.88      0.86      1035
           1       0.63      0.55      0.58       374

    accuracy                           0.79      1409
   macro avg       0.73      0.71      0.72      1409
weighted avg       0.79      0.79      0.79      1409

AUC Score: 0.8373272365599731


In [19]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print("Confusion Matrix:")
print(cm_xgb)

Confusion Matrix:
[[913 122]
 [170 204]]


##  3. LightGBM (Light Gradient Boosting Machine)

###  How it Works:
LightGBM is also a **gradient boosting framework**, but it uses a more optimized tree-building strategy called **leaf-wise growth**, making it faster and more efficient than traditional boosting methods.

###  Importance in This Project:
- Very fast training speed, even on large datasets  
- Lower memory usage compared to other boosting models  
- High accuracy similar to XGBoost  
- Handles categorical patterns efficiently  

###  Why We Use It:
LightGBM is ideal for **scalable production systems**, especially when dealing with large-scale customer data and frequent retraining (such as weekly scoring pipelines).


In [20]:
!pip install lightgbm

Defaulting to user installation because normal site-packages is not writeable

You should consider upgrading via the 'C:\Program Files\Python310\python.exe -m pip install --upgrade pip' command.


In [21]:
from lightgbm import LGBMClassifier

In [22]:
lgb_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42
)

In [23]:
lgb_model.fit(X_train, y_train)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4139
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000946 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 637
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265353 -> initscore=-1.018328
[LightGBM] [Info] Start training from score -1.018328


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [24]:
y_pred_lgb = lgb_model.predict(X_test)
y_prob_lgb = lgb_model.predict_proba(X_test)[:, 1]

In [25]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
print("LightGBM Results:\n")
print(classification_report(y_test, y_pred_lgb))
print("AUC Score:", roc_auc_score(y_test, y_prob_lgb))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lgb))

LightGBM Results:

              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1035
           1       0.63      0.53      0.57       374

    accuracy                           0.79      1409
   macro avg       0.73      0.71      0.72      1409
weighted avg       0.78      0.79      0.78      1409

AUC Score: 0.8322521894133148

Confusion Matrix:
[[917 118]
 [177 197]]


In [26]:
import joblib

joblib.dump(X.columns, "../models/model_features.pkl")

['../models/model_features.pkl']